# 09 — MVP de bout en bout

Ce notebook utilise le package de production du dépôt. Il montre le chemin complet entre un corpus, l'index hybride et une réponse citée.

Le corpus ci-dessous est fictif pour que le notebook reste immédiatement exécutable. Pour des documents réels, utiliser le catalogue officiel et valider chaque version avant indexation.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from morocco_legal_rag.chunking import chunk_page
from morocco_legal_rag.generation import ExtractiveGenerator
from morocco_legal_rag.retrieval import HybridRetriever
from morocco_legal_rag.service import LegalRAGService

pages = [
    ("DEMO-A", "Document pédagogique A", 1, "Le délai pédagogique de traitement est de dix jours ouvrables."),
    ("DEMO-B", "Document pédagogique B", 2, "Une correction nécessite une demande écrite et un justificatif rectifié."),
    ("DEMO-C", "Document pédagogique C", 4, "Le dépôt pédagogique est disponible au guichet de démonstration."),
]
chunks = [
    chunk
    for document_id, title, page, text in pages
    for chunk in chunk_page(
        document_id=document_id,
        title=title,
        page=page,
        text=text,
        language="fr",
        size=40,
        overlap=5,
    )
]
len(chunks)

In [ ]:
retriever = HybridRetriever()
retriever.fit(chunks)
service = LegalRAGService(retriever, ExtractiveGenerator())

response = service.ask("Quel est le délai de traitement ?", language="fr", top_k=2)
print(response.answer)
response.citations

## Passer au modèle multilingue

Après `pip install -e ".[ml]"`, remplacer le baseline par :

```python
from morocco_legal_rag.retrieval import SentenceTransformerEncoder
retriever = HybridRetriever(encoder=SentenceTransformerEncoder())
```

Le reste du pipeline demeure identique, ce qui rend les expériences comparables.